# Notebook 01 — Project Description
## Amazon LA Delivery Failure Prediction

**Program:** Correlation One — Data Analytics (DANA), Week 12 Final Portfolio Project  
**Date:** April 2026

---

## Overview

This notebook introduces the business context, problem statement, and dataset for the Amazon LA last-mile delivery failure prediction project.

The goal is to build a predictive system that flags high-risk packages **before dispatch** — enabling proactive intervention and reducing both cost and customer experience degradation from failed deliveries.

## 1. Industry Context: Amazon LA Last-Mile Logistics

Amazon LA's last-mile delivery network faces significant operational complexity:

| Challenge | Impact |
|---|---|
| Mixed urban/suburban geography | Route efficiency varies dramatically |
| Multi-carrier dependency | 4 carriers with different SLA performance |
| Mediterranean weather events | Flooding, Levante winds, summer heat |
| Municipal access restrictions | Madrid ZBE, Los Angeles Superilles |
| Spanish labor regulations | Constrained delivery windows |

**Key Amazon Operations Metrics:**
- **DPMO**: Defects Per Million Opportunities — primary failure rate KPI
- **DEA**: Delivery Experience Accuracy — promise fulfillment metric  
- **VOC**: Voice of Customer — downstream satisfaction signal
- **OTIF**: On-Time In-Full — SLA compliance metric

## 2. Problem Statement

### Current State (Reactive)
Amazon LA operations teams currently analyze delivery failures **after the fact** through weekly DPMO reporting. This reactive approach means:

1. Failure costs are **already incurred** before intervention
2. Customer experience scores (VOC) **already degraded** before corrective action
3. Route optimization decisions made **without failure probability information**

### Target State (Proactive)
A **pre-dispatch scoring model** estimates failure probability for each package given:
- Package characteristics (type, condition)
- Carrier assignment
- Route attributes
- Operational flags
- Environmental conditions

### Business Question
> **Given what we know at dispatch time, will this package fail to be delivered?**

### Financial Impact
- Avg cost per failed delivery: **$17** (redelivery + CS contact)
- Dataset failure rate: **~0.70%** → ~25 failures in 3,559 packages
- Model catching 80% of failures could prevent **hundreds of failures daily**

In [ ]:
# Load libraries and preview dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = 'white'

# Load training data
df = pd.read_csv('../data/packages_train.csv')
print(f'Dataset shape: {df.shape}')
print(f'Failure rate: {df["delivery_failed"].mean():.1%}')
df.head(10)

## 3. Dataset Schema

| Column | Type | Description | Role |
|---|---|---|---|
| package_id | string | Unique identifier (PKG-ES-XXXXXXX) | ID only |
| package_type | categorical | standard / high_value / fragile / locker / large | Feature |
| shift | categorical | morning / afternoon / night | Feature |
| carrier | categorical | carrier_A=Amazon, B=SEUR, C=DHL, D=Regional Carrier | Feature |
| route_distance_km | float | Route distance (2–85 km) | Feature |
| packages_in_route | int | Total packages in driver route (15–120) | Feature |
| double_scan | binary | Scan error flag | Feature |
| short_service_time | binary | Planned service time under 25 seconds | Feature |
| delivery_failed | binary | Package damaged when received at FC | Feature |
| weather_risk | categorical | low / medium / high | Feature |
| cr_number_missing | binary | Missing customer reference number | Feature |
| days_in_fc | int | Days in fulfillment center (0–12) | Feature |
| delivery_failed | binary | **TARGET**: 1=failed, 0=delivered | Target |

In [ ]:
# Data types and basic statistics
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Numerical Summary ===')
df.describe()

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['delivery_failed'].value_counts()
labels = ['Delivered (0)', 'Failed (1)']
colors = ['#146EB4', '#FF9900']
axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Target Variable Distribution', fontsize=13, fontweight='bold', color='#232F3E')
axes[0].set_ylabel('Package Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold', color='#232F3E')

# Pie chart
axes[1].pie([counts[0], counts[1]], labels=['Delivered', 'Failed'],
             colors=colors, autopct='%1.1f%%', startangle=90,
             wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Delivery Success Rate', fontsize=13, fontweight='bold', color='#232F3E')

plt.suptitle('Amazon LA — Package Delivery Outcomes (Training Set, n=5,000)',
             fontsize=14, fontweight='bold', color='#232F3E', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Delivered: {counts[0]:,} ({counts[0]/len(df):.1%})')
print(f'Failed:    {counts[1]:,} ({counts[1]/len(df):.1%})')

## 4. Business Value Summary

This project delivers value at three levels:

### Operational Value
- Pre-dispatch failure scoring enables **proactive intervention** for high-risk packages
- Operations managers can redirect packages from carrier_D to carrier_A for long routes
- Damaged packages can be held for QA before dispatch

### Financial Value
- Catching 80% of failures (model recall) at $8/failure = **significant per-shift savings**
- ROI on model deployment pays back within weeks at Amazon scale

### Strategic Value
- Foundation for **real-time WMS integration** for automated dispatch scoring
- Data asset for **carrier SLA renegotiation** (carrier_D performance evidence)
- Building block for **multi-attempt failure type prediction** (wrong address vs. customer unavailable)

---
*See `reports/project_description.md` for the full written report.*